In [1]:
import json
import re
from typing import Dict, List
from tqdm import tqdm

In [2]:
user_data = []

with open("mpchat_hybrid_sentiment_updated.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        user_data.append(json.loads(line))

print(f"Loaded {len(user_data)} samples")

Loaded 7007 samples


In [3]:
def split_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"[.!?]", text) if s.strip()]

In [15]:
def guard_and_clamp(parsed: dict, max_sentences: int = 3) -> dict:
    """
    parsed: already-parsed JSON dict with keys {facts, style}
    """
    memory = {}

    for key in ["facts", "style"]:
        text = parsed.get(key, "")
        if not isinstance(text, str):
            text = ""

        sentences = split_sentences(text)
        memory[key] = ". ".join(sentences[:max_sentences])

    return memory

In [16]:
import re

def split_sentences(text: str):
    return [s.strip() for s in re.split(r"[.!?]", text) if s.strip()]

In [5]:
def compress(text: str, max_sentences: int = 5) -> str:
    sentences = split_sentences(text)
    return ". ".join(sentences[:max_sentences])

In [77]:
def build_persona_prompt(speaker, previous_memory, user_reply, context_meta):
    return (
        "You are incrementally updating a long-term user persona.\n\n"

        f"Speaker: {speaker}\n\n"

        "Current persona memory:\n"
        f"{json.dumps(previous_memory)}\n\n"

        "New user utterance:\n"
        f"{user_reply}\n\n"

        "Interaction signals (use only as soft hints):\n"
        f"- Sentiment toward user: {context_meta['sentiment']}\n"
        f"- Emotional valence score: {context_meta['alice_sentiment']}\n"
        f"- Conversational alignment score: {context_meta['similarity']}\n"
        f"- Positive vs negative tone: {context_meta['pos_score']} / {context_meta['neg_score']}\n\n"

        "Rules:\n"
        "- Update memory only if the trait is likely to persist across interactions.\n"
        "- You MAY add a trait tentatively if it is clearly expressed and not highly specific.\n"
        "- Facts: stable self-identification or long-term roles only.\n"
        "- Style: tone or expressiveness.\n"
        "- Interests: HIGH-LEVEL recurring topics (abstract, no brands, no locations).\n"
        "- Do NOT add speculative or weak signals.\n"
        "- If nothing should change, return the memory unchanged.\n\n"

        "Respond with JSON ONLY in this exact format:\n"
        '{"facts": "", "style": "", "interests": []}'
    )

In [18]:
def stabilize_memory(prev: dict, new: dict) -> dict:
    """
    Prevents accidental overwrites or hallucinated updates.
    """
    out = {}

    for key in ["facts", "style"]:
        if new[key].strip() == "":
            out[key] = prev[key]
        else:
            out[key] = new[key]

    return out

In [23]:
def build_interest_prompt(speaker, user_reply):
    return (
        f"You are analyzing a single user reply.\n"
        f"Extract possible INTEREST TOPICS the user is engaging with.\n\n"

        f"Speaker: {speaker}\n\n"
        f"User reply:\n"
        f"{user_reply}\n\n"

        f"Rules:\n"
        f"- Interests are recurring domains (e.g., coffee brewing, photography).\n"
        f"- Do NOT infer preferences or emotions.\n"
        f"- Do NOT invent interests.\n"
        f"- If no clear interest is present, return an empty list.\n\n"

        f"Respond with JSON only in this format:\n"
        f'{{"interest_candidates": ["..."]}}'
    )

In [70]:
def build_trajectory_prompt(previous_trajectory, context_text, user_reply):
    return (
        "You are summarizing what happened in this interaction.\n\n"

        "Previous summary:\n"
        f"{previous_trajectory}\n\n"

        "Context messages:\n"
        f"{context_text}\n\n"

        "User reply:\n"
        f"{user_reply}\n\n"

        "Rules:\n"
        "- Focus on topic and intent only.\n"
        "- Do NOT infer personality or long-term traits.\n"
        "- Keep summary factual and concise.\n\n"

        "Write updated summary:"
    )

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir="D:/hf_offload"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir="D:/hf_offload",
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [10]:
def call_llm(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9
) -> str:
    """
    Low-level causal generation wrapper for Mistral-7B.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove prompt tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return output_text.strip()


In [21]:
def call_llm_debug(prompt: str, tag: str = "", max_new_tokens: int = 128) -> str:
    print("\n" + "-" * 80)
    print(f"[LLM INPUT PROMPT] {tag}")
    print("-" * 80)
    print(prompt)
    print("-" * 80)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    print(f"Token count: {inputs['input_ids'].shape[-1]}")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # keep first JSON only
    start, end = text.find("{"), text.find("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end+1]

    print("\n[LLM RAW OUTPUT]")
    print(text)
    print("-" * 80)

    return text.strip()

In [69]:
import re

def safe_parse_memory(raw_text):
    try:
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if not match:
            return {"facts": "", "style": "", "interests": []}

        parsed = json.loads(match.group())

        return {
            "facts": parsed.get("facts", "").strip(),
            "style": parsed.get("style", "").strip(),
            "interests": parsed.get("interests", []) if isinstance(parsed.get("interests", []), list) else []
        }
    except Exception:
        return {"facts": "", "style": "", "interests": []}

In [67]:
def compress(text, max_sentences=3):
    if not text:
        return ""

    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return " ".join(sentences[:max_sentences])

In [13]:
def debug_tokenization(prompt: str):
    tokens = tokenizer(prompt, return_tensors="pt")
    print(f"\nToken count: {tokens['input_ids'].shape[-1]}")
    print("First 40 tokens:")
    print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0][:40]))

In [25]:
def update_interest_evidence(candidates):
    for c in candidates:
        interest_evidence[c.lower()] += 1

In [27]:
from collections import Counter

interest_evidence = Counter()
INTEREST_THRESHOLD = 3

In [28]:
def promote_interests_if_ready(memory, threshold=INTEREST_THRESHOLD):
    confirmed = [
        i for i, count in interest_evidence.items()
        if count >= threshold
    ]

    memory["interests"] = sorted(confirmed)
    return memory

In [24]:
def parse_interest_candidates(raw_text: str):
    try:
        obj = json.loads(raw_text)
        return obj.get("interest_candidates", [])
    except Exception:
        return []

In [ ]:
def parse_style_candidates(raw_text: str):
    try:
        obj = json.loads(raw_text)
        return obj.get("style_candidates", [])
    except Exception:
        return []

In [ ]:
def update_style_evidence(candidates):
    for c in candidates:
        style_evidence[c.lower()] += 1

In [ ]:
def promote_style_if_ready(memory, threshold=STYLE_THRESHOLD):
    promoted = [
        s for s, count in style_evidence.items()
        if count >= threshold
    ]

    if promoted:
        memory["style"] = ", ".join(sorted(promoted))

    return memory

In [36]:
from collections import Counter

style_evidence = Counter()
interest_evidence = Counter()

STYLE_THRESHOLD = 3
INTEREST_THRESHOLD = 3

In [37]:
print(build_persona_prompt("nickeymousee", {"facts": "", "style": "", "interests": []}, "test reply"))

You are maintaining long-term persona memory for a user.

Speaker: nickeymousee

Existing persona memory:
{"facts": "", "style": "", "interests": []}

Latest user reply:
test reply

Rules:
- Update memory only if the signal is likely to persist across conversations.
- Style = tone, expressiveness, formality.
- Interests = recurring topics the user engages with.
- Do NOT infer one-off preferences.
- If nothing should change, return the memory unchanged.

Respond with JSON only in this exact format:
{"facts": "", "style": "", "interests": []}


In [73]:
user_data_filtered = [
    s for s in user_data
    if s["target_speaker"] == TARGET_USER
]

In [75]:
len(user_data_filtered)

6

In [78]:
TARGET_USER = "nickeymousee"

memory = {
    "facts": "",
    "style": "",
    "interests": []
}
trajectory = ""

for i, sample in enumerate(user_data_filtered):
    print("\n" + "=" * 80)
    print(f"STEP {i+1}")
    print(f"Turn index: {sample['s_node']['turn_index']}")

    # Context text (trajectory only)
    context_text = "\n".join(
        f"{c['speaker_id']}: {c['utterance']}"
        for c in sample["context"]
    )

    user_reply = sample["s_node"]["utterance"]

    # ---- Extract ALL usable signals ----
    ctx = sample["context"][0]

    context_meta = {
        "sentiment": ctx.get("sentiment_towards_target", ""),
        "alice_sentiment": ctx.get("debug", {}).get("alice_sentiment", 0.0),
        "similarity": ctx.get("debug", {}).get("similarity", 0.0),
        "pos_score": ctx.get("debug", {}).get("pos_score", 0.0),
        "neg_score": ctx.get("debug", {}).get("neg_score", 0.0)
    }

    # ----------------------------------
    # PERSONA MEMORY UPDATE
    # ----------------------------------
    persona_prompt = build_persona_prompt(
        TARGET_USER,
        memory,
        user_reply,
        context_meta
    )

    raw_persona = call_llm_debug(
        persona_prompt,
        tag="PERSONA MEMORY UPDATE",
        max_new_tokens=128
    )

    parsed_persona = safe_parse_memory(raw_persona)

    # Stabilize memory
    for key in ["facts", "style", "interests"]:
        if parsed_persona[key]:
            memory[key] = parsed_persona[key]

    # ----------------------------------
    # TRAJECTORY UPDATE
    # ----------------------------------
    trajectory_prompt = build_trajectory_prompt(
        trajectory,
        context_text,
        user_reply
    )

    raw_traj = call_llm_debug(
        trajectory_prompt,
        tag="TRAJECTORY UPDATE",
        max_new_tokens=128
    )

    trajectory = compress(raw_traj, 3)

    print("\nUPDATED MEMORY:", memory)
    print("UPDATED TRAJECTORY:", trajectory)


STEP 1
Turn index: 2

--------------------------------------------------------------------------------
[LLM INPUT PROMPT] PERSONA MEMORY UPDATE
--------------------------------------------------------------------------------
You are incrementally updating a long-term user persona.

Speaker: nickeymousee

Current persona memory:
{"facts": "", "style": "", "interests": []}

New user utterance:
which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool

Interaction signals (use only as soft hints):
- Sentiment toward user: positive
- Emotional valence score: 0.7177
- Conversational alignment score: 0.35043150186538696
- Positive vs negative tone: 1.0 / 0.5

Rules:
- Update memory only if the trait is likely to persist across interactions.
- You MAY add a trait tentatively if it is clearly expressed and not highly specific.
- Facts: stable self-identification or long-term roles only.
- Style: tone or expressiveness.
- Interests: 